# 02 — Fast Model: TF-IDF + CatBoost

Обучаем модель для тарифа **fast** (< 100ms, 1 кредит).

Артефакты на выходе:
- `models/tfidf.pkl` — TfidfVectorizer
- `models/catboost_model.cbm` — CatBoostClassifier
- `models/fast_metrics.json` — метрики

**Зависит от:** `data/train.csv`, `data/val.csv`, `data/test.csv` из ноутбука 01.

In [1]:
!pip install catboost scikit-learn pandas numpy -q

In [2]:
import pandas as pd
import numpy as np
import pickle, json, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score, accuracy_score
from catboost import CatBoostClassifier
import scipy.sparse as sp

os.makedirs('models', exist_ok=True)

In [3]:
train = pd.read_csv('/kaggle/input/datasets/vegeradaria21/service-data/data/train.csv')
val   = pd.read_csv('/kaggle/input/datasets/vegeradaria21/service-data/data/val.csv')
test  = pd.read_csv('/kaggle/input/datasets/vegeradaria21/service-data/data/test (1).csv')

with open('/kaggle/input/datasets/vegeradaria21/service-data/data/label2id.json') as f:
    id2label = {int(v): k for k, v in json.load(f).items()}

print('Train:', len(train), '| Val:', len(val), '| Test:', len(test))
print('Классы:', train['label'].nunique())

Train: 11492 | Val: 2031 | Test: 2968
Классы: 4


## 1. Предобработка текста

In [4]:
import re

def preprocess(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train['text_clean'] = train['text'].apply(preprocess)
val['text_clean']   = val['text'].apply(preprocess)
test['text_clean']  = test['text'].apply(preprocess)

print('Пример:', train['text_clean'].iloc[0])

Пример: давай пропустим встречу в это время


## 2. TF-IDF признаки + мета-признаки

**Важно:** функция `extract_meta` должна работать одинаково при обучении и инференсе.
Все мета-признаки считаются только по исходному тексту (не очищенному).

In [5]:
META_FEATURES = 4  # фиксируем количество — важно для инференса

def extract_meta(texts_series: pd.Series) -> np.ndarray:
    """Извлекает 4 числовых мета-признака из сырых текстов.
    
    Порядок признаков строго фиксирован (документировано в fast_metrics.json):
    0: длина текста в символах
    1: доля заглавных букв
    2: количество слов
    3: количество знаков вопроса и восклицания
    """
    texts = texts_series.fillna('').astype(str)
    char_len   = texts.str.len().values.astype(float)
    upper_ratio = texts.apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)).values
    word_count  = texts.str.split().apply(len).values.astype(float)
    punct_count = texts.str.count(r'[?!]').values.astype(float)
    return np.column_stack([char_len, upper_ratio, word_count, punct_count])

def extract_meta_single(text: str) -> np.ndarray:
    """Версия для одного текста при инференсе — возвращает (1, 4)."""
    return extract_meta(pd.Series([text]))

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    analyzer='word',
)
tfidf.fit(train['text_clean'])

X_train = sp.hstack([tfidf.transform(train['text_clean']), extract_meta(train['text'])])
X_val   = sp.hstack([tfidf.transform(val['text_clean']),   extract_meta(val['text'])])
X_test  = sp.hstack([tfidf.transform(test['text_clean']),  extract_meta(test['text'])])

y_train = train['label'].values
y_val   = val['label'].values
y_test  = test['label'].values

print('X_train shape:', X_train.shape)
print('Мета-признаков:', META_FEATURES, '(фиксировано)')

X_train shape: (11492, 9454)
Мета-признаков: 4 (фиксировано)


## 3. Обучение CatBoost

In [6]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=50,
    early_stopping_rounds=30
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
)

0:	learn: 0.5150540	test: 0.5361891	best: 0.5361891 (0)	total: 330ms	remaining: 2m 44s
50:	learn: 0.7475635	test: 0.7543082	best: 0.7543082 (50)	total: 8.2s	remaining: 1m 12s
100:	learn: 0.8016011	test: 0.8060069	best: 0.8060069 (100)	total: 16.1s	remaining: 1m 3s
150:	learn: 0.8312739	test: 0.8252093	best: 0.8252093 (149)	total: 23.9s	remaining: 55.3s
200:	learn: 0.8481552	test: 0.8389956	best: 0.8389956 (200)	total: 31.7s	remaining: 47.1s
250:	learn: 0.8594675	test: 0.8483506	best: 0.8498277 (247)	total: 39.5s	remaining: 39.2s
300:	learn: 0.8660808	test: 0.8572132	best: 0.8581979 (289)	total: 47.3s	remaining: 31.3s
350:	learn: 0.8737383	test: 0.8621369	best: 0.8621369 (349)	total: 55.1s	remaining: 23.4s
400:	learn: 0.8793073	test: 0.8655835	best: 0.8655835 (385)	total: 1m 2s	remaining: 15.5s
450:	learn: 0.8836582	test: 0.8700148	best: 0.8700148 (449)	total: 1m 10s	remaining: 7.68s
499:	learn: 0.8873129	test: 0.8734613	best: 0.8744461 (480)	total: 1m 18s	remaining: 0us

bestTest = 0.8

CatBoostClassifier(depth=6, early_stopping_rounds=30, eval_metric='Accuracy', iterations=500, learning_rate=0.1, loss_function='MultiClass', random_seed=42, verbose=50)

## 4. Оценка на тесте

In [8]:
y_pred = model.predict(X_test).flatten()
acc    = accuracy_score(y_test, y_pred)
f1     = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy: {acc:.4f}')
print(f'F1 macro: {f1:.4f}')
print()
print(classification_report(
    y_test, y_pred,
    target_names=[id2label[i] for i in range(4)]
))

Accuracy: 0.8608
F1 macro: 0.8709

                    precision    recall  f1-score   support

account_management       0.92      0.84      0.88      1017
     entertainment       0.96      0.75      0.84       614
   general_inquiry       0.76      0.94      0.84      1117
   technical_issue       0.99      0.85      0.91       220

          accuracy                           0.86      2968
         macro avg       0.91      0.85      0.87      2968
      weighted avg       0.88      0.86      0.86      2968



## 5. Сохранение артефактов

In [9]:
# TF-IDF
with open('models/tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# CatBoost
model.save_model('models/catboost_model.cbm')

In [10]:
# Метрики + мета-информация о признаках (критично для worker)
metrics = {
    'accuracy': round(float(acc), 4),
    'f1_macro': round(float(f1), 4),
    'model': 'CatBoost + TF-IDF',
    'n_train': len(train),
    'n_test': len(test),
    'classes': [id2label[i] for i in range(4)],
    'meta_features': [
        'char_len', 'upper_ratio', 'word_count', 'punct_count'
    ],
    'meta_features_count': META_FEATURES,
    'tfidf_max_features': 50000,
    'tfidf_ngram_range': [1, 2],
}
with open('models/fast_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('Сохранено:')
print('  models/tfidf.pkl')
print('  models/catboost_model.cbm')
print('  models/fast_metrics.json')
print(f'\nF1 macro: {f1:.4f} | Accuracy: {acc:.4f}')

Сохранено:
  models/tfidf.pkl
  models/catboost_model.cbm
  models/fast_metrics.json

F1 macro: 0.8709 | Accuracy: 0.8608


## 6. Проверка скорости инференса + корректности пайплайна

In [11]:
import time


with open('models/tfidf.pkl', 'rb') as f:
    loaded_tfidf = pickle.load(f)

loaded_model = CatBoostClassifier()
loaded_model.load_model('models/catboost_model.cbm')

CatBoostClassifier(depth=6, eval_metric='Accuracy', iterations=500, learning_rate=0.1, loss_function='MultiClass', od_wait=30, random_seed=42, use_best_model=True, verbose=50)

In [12]:
def predict_fast(text: str):
    """Полный пайплайн инференса — точная копия worker/ml/fast_predictor.py"""
    clean = preprocess(text)
    tfidf_feat = loaded_tfidf.transform([clean])           
    meta_feat  = extract_meta_single(text)                   
    features   = sp.hstack([tfidf_feat, meta_feat])      
    proba = loaded_model.predict_proba(features)[0]        
    idx = int(np.argmax(proba))
    return id2label[idx], float(proba[idx])

In [13]:
test_texts = [
    'Хочу вернуть товар',
    'Не работает приложение',
    'Списали деньги дважды',
    'Взломали мой аккаунт',
    'Требую руководителя',
    'Как отключить уведомления?',
    'Хочу пожаловаться на качество',
    'Где можно посмотреть статус заказа',
]

times = []
for text in test_texts:
    t0 = time.perf_counter()
    intent, conf = predict_fast(text)
    elapsed = (time.perf_counter() - t0) * 1000
    times.append(elapsed)
    status = 'DONE ' if elapsed < 100 else 'СЛИШКОМ МЕДЛЕННО'
    print(f'[{elapsed:5.1f}ms] {status}  {text!r:40s} → {intent} ({conf:.3f})')

print(f'\nСреднее: {np.mean(times):.1f}ms | Макс: {np.max(times):.1f}ms')
print(f'Цель: < 100ms — {"ВЫПОЛНЕНО" if np.mean(times) < 100 else "НЕ ВЫПОЛНЕНО"}')

[ 13.9ms] DONE   'Хочу вернуть товар'                     → general_inquiry (0.459)
[ 13.4ms] DONE   'Не работает приложение'                 → general_inquiry (0.520)
[ 11.7ms] DONE   'Списали деньги дважды'                  → general_inquiry (0.576)
[ 10.4ms] DONE   'Взломали мой аккаунт'                   → general_inquiry (0.454)
[ 10.4ms] DONE   'Требую руководителя'                    → general_inquiry (0.568)
[ 10.5ms] DONE   'Как отключить уведомления?'             → general_inquiry (0.664)
[ 10.7ms] DONE   'Хочу пожаловаться на качество'          → general_inquiry (0.467)
[ 11.1ms] DONE   'Где можно посмотреть статус заказа'     → general_inquiry (0.595)

Среднее: 11.5ms | Макс: 13.9ms
Цель: < 100ms — ВЫПОЛНЕНО


## 7. Куда кладём файлы в проекте

После обучения скачай папку `models/` и положи:

```
worker/ml/models/
├── tfidf.pkl
├── catboost_model.cbm
└── fast_metrics.json
```

Или смонтируй как Kaggle Dataset и укажи пути в `.env` через переменную `FAST_MODEL_PATH`.